Use the SimLM model to help create a score based on the relevant data presented by the Ardook

Model Link: https://huggingface.co/intfloat/simlm-msmarco-reranker

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
#Imports and Installation
!pip install transformers

import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BatchEncoding, PreTrainedTokenizerFast
from transformers.modeling_outputs import SequenceClassifierOutput
import torch
import pandas as pd
import numpy as np


In [28]:
def encode(tokenizer: PreTrainedTokenizerFast,
           query: str, passage: str, title: str = '-') -> BatchEncoding:
    return tokenizer(query,
                     text_pair='{}: {}'.format(title, passage),
                     max_length=192,
                     padding=True,
                     truncation=True,
                     return_tensors='pt')

tokenizer = AutoTokenizer.from_pretrained('intfloat/simlm-msmarco-reranker')
model = AutoModelForSequenceClassification.from_pretrained('intfloat/simlm-msmarco-reranker')
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ElectraForSequenceClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): L

In [29]:
FILE_PATH  = "/content/drive/MyDrive/Copy of Ardook_Data_All_Grant_Recommendations_UPDATED_v3.xlsx"   # path after uploading data
SHEET_NAME = "Grant Tracker"
TEXT_COL   = "Core Requirements"

In [30]:
df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
df.columns = df.columns.str.strip()
print(df.shape)
print("Empty requirement cells:", df[TEXT_COL].isna().sum())

# Creates the table
data = df[["Priority / Deadline Order", "Program", TEXT_COL, "Fit"]].copy()
data[TEXT_COL] = data[TEXT_COL].fillna("").astype(str).str.strip()
data.head()

(37, 13)
Empty requirement cells: 0


,Priority / Deadline Order,Program,Core Requirements,Fit
0,1,Women TechEU 2026–2028,Woman founder/co-founder in top management; wo...,Excellent
1,2,Tara’s Circle 2026,Women-led; international; seed-stage; defined/...,Excellent
2,3,Her Agenda Breakthrough Grant,Woman entrepreneur; operating business; newsle...,Strong
3,4,Hey Helen Grant (Visionaries),Woman; ≥50% ownership; U.S. resident; U.S. for...,Strong
4,5,WomensNet Amber Grant / Startup Grant,Women-owned business; Startup Grant track for ...,Strong


In [33]:
#Input Comparisons
scores = []
for index, row in data.iterrows():
  with torch.no_grad():
      bat_di = encode(tokenizer, 'List the scores of the grant available to women entrepeneurs based on the fit', str(row[TEXT_COL]), str(row['Fit']))
      outputs: SequenceClassifierOutput = model(**bat_di, return_dict=True)
      results = outputs.logits[0].item()
      print(results)
      scores.append(results)

      #print(outputs.logits[0].item())
      #results.append(outputs.logits[0].item())

data[scores] = results


-7.44921875
-7.43359375
-7.359375
-7.37890625
-7.109375
-7.4453125
-7.234375
-7.36328125
-7.359375
-7.50390625
-7.609375
-7.41796875
-7.44921875
-7.4296875
-7.21484375
-7.34765625
-7.51953125
-7.3984375
-7.52734375
-7.59765625
-7.45703125
-7.37109375
-7.30078125
-7.484375
-7.59765625
-7.50390625
-7.42578125
-7.453125
-7.44921875
-7.4375
-7.453125
-7.4140625
-7.25390625
-7.4765625
-7.39453125
-7.26171875
-7.296875
